In [0]:
import requests

url = "https://api.coingecko.com/api/v3/ping"

response = requests.get(url, timeout=10)

print("Status:", response.status_code)
print("Response:", response.text)

In [0]:
import requests

url = "https://api.coingecko.com/api/v3/coins/markets"

params = {
    "vs_currency": "usd",
    "per_page": 10,
    "page": 1
}

response = requests.get(
    url,
    params=params,
    timeout=10
)

print("Status:", response.status_code)
print("Response:", response.text[:2000])

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    LongType,
    TimestampType
)

coin_bronze_schema = StructType([
    StructField("coin_id", StringType(), False),
    StructField("symbol", StringType(), False),
    StructField("name", StringType(), False),
    StructField("current_price", DoubleType(), True),
    StructField("market_cap", LongType(), True),
    StructField("market_cap_rank", LongType(), True),
    StructField("total_volume", DoubleType(), True),
    StructField("high_24h", DoubleType(), True),
    StructField("low_24h", DoubleType(), True),
    StructField("price_change_24h", DoubleType(), True),
    StructField("price_change_percentage_24h", DoubleType(), True),
    StructField("circulating_supply", DoubleType(), True),
    StructField("total_supply", DoubleType(), True),
    StructField("max_supply", DoubleType(), True),
    StructField("ath", DoubleType(), True),
    StructField("atl", DoubleType(), True),
    StructField("last_updated", TimestampType(), True),
    StructField("source_system", StringType(), False),
    StructField("ingestion_timestamp", TimestampType(), False)
])

print("Coin Bronze schema created successfully")

In [0]:
import requests

url = "https://api.coingecko.com/api/v3/coins/markets"

params = {
    "vs_currency": "usd",
    "per_page": 10,
    "page": 1
}

response = requests.get(
    url,
    params=params,
    timeout=10
)

coins_data = response.json()

print("Records received:", len(coins_data))

In [0]:
coins_df = spark.createDataFrame(
    coins_data,
    schema=coin_bronze_schema
)

print("Spark DataFrame created successfully")

In [0]:
for i, coin in enumerate(coins_data):
    for key, value in coin.items():
        if value is None:
            print("Record:", i, "| Field:", key, "| Value:", value)

In [0]:
coin_rows = []

for coin in coins_data:
    coin_rows.append((
        coin.get("id"),
        coin.get("symbol"),
        coin.get("name"),
        coin.get("current_price"),
        coin.get("market_cap"),
        coin.get("market_cap_rank"),
        coin.get("total_volume"),
        coin.get("high_24h"),
        coin.get("low_24h"),
        coin.get("price_change_24h"),
        coin.get("price_change_percentage_24h"),
        coin.get("circulating_supply"),
        coin.get("total_supply"),
        coin.get("max_supply"),
        coin.get("ath"),
        coin.get("atl"),
        coin.get("last_updated")
    ))

print("Rows prepared:", len(coin_rows))

In [0]:
coins_df = spark.createDataFrame(
    coin_rows,
    schema=coin_bronze_schema
)

print("Spark DataFrame created successfully")

In [0]:
from pyspark.sql.functions import current_timestamp

coin_rows = []

for coin in coins_data:
    coin_rows.append((
        coin.get("id"),
        coin.get("symbol"),
        coin.get("name"),
        coin.get("current_price"),
        coin.get("market_cap"),
        coin.get("market_cap_rank"),
        coin.get("total_volume"),
        coin.get("high_24h"),
        coin.get("low_24h"),
        coin.get("price_change_24h"),
        coin.get("price_change_percentage_24h"),
        coin.get("circulating_supply"),
        coin.get("total_supply"),
        coin.get("max_supply"),
        coin.get("ath"),
        coin.get("atl"),
        coin.get("last_updated"),
        "CoinGecko",
        None
    ))

print("Values per row:", len(coin_rows[0]))

In [0]:
coins_df = spark.createDataFrame(
    coin_rows,
    schema=coin_bronze_schema
)

print("Spark DataFrame created successfully")

In [0]:
from datetime import datetime, timezone

coin_rows = []

for coin in coins_data:

    last_updated = None

    if coin.get("last_updated"):
        last_updated = datetime.fromisoformat(
            coin["last_updated"].replace("Z", "+00:00")
        )

    ingestion_timestamp = datetime.now(timezone.utc)

    coin_rows.append((
        coin.get("id"),
        coin.get("symbol"),
        coin.get("name"),
        coin.get("current_price"),
        coin.get("market_cap"),
        coin.get("market_cap_rank"),
        coin.get("total_volume"),
        coin.get("high_24h"),
        coin.get("low_24h"),
        coin.get("price_change_24h"),
        coin.get("price_change_percentage_24h"),
        coin.get("circulating_supply"),
        coin.get("total_supply"),
        coin.get("max_supply"),
        coin.get("ath"),
        coin.get("atl"),
        last_updated,
        "CoinGecko",
        ingestion_timestamp
    ))

print("Rows prepared:", len(coin_rows))
print("Values per row:", len(coin_rows[0]))

In [0]:
coins_df = spark.createDataFrame(
    coin_rows,
    schema=coin_bronze_schema
)

print("Spark DataFrame created successfully")

In [0]:
coins_df.show(truncate=False)

In [0]:
target_table = "bronze_coin_master"

coins_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(target_table)

print(f"Data written successfully to {target_table}")

In [0]:
spark.sql("""
SELECT *
FROM bronze_coin_master
""").show(truncate=False)

In [0]:
spark.sql("""
SELECT COUNT(*) AS total_records
FROM bronze_coin_master
""").show()

In [0]:
%sql
SELECT
    COUNT(*) AS total_records,

    COUNT(DISTINCT coin_id) AS unique_coin_ids,

    COUNT(*) - COUNT(DISTINCT coin_id) AS duplicate_records,

    COUNT(*) FILTER (WHERE coin_id IS NULL) AS null_coin_ids,

    COUNT(*) FILTER (WHERE symbol IS NULL) AS null_symbols,

    COUNT(*) FILTER (WHERE name IS NULL) AS null_names,

    COUNT(*) FILTER (WHERE current_price IS NULL) AS null_prices,

    COUNT(*) FILTER (WHERE current_price <= 0) AS invalid_price_records,

    COUNT(*) FILTER (WHERE last_updated IS NULL) AS null_last_updated,

    COUNT(*) FILTER (WHERE ingestion_timestamp IS NULL) AS null_ingestion_timestamp

FROM bronze_coin_master;